## 1. Intuition

PCA only captures **linear** structure — it finds the directions of
maximum variance, full stop. Real data (images, embeddings, gene
expression) often lies on a **curved manifold**: think of a Swiss roll —
PCA would just flatten it destructively, but the "correct" 2D
representation should unroll it. t-SNE and UMAP are built to preserve
*local neighborhood structure* (which points are close to which) even
when the global geometry is nonlinear.

## 2. t-SNE — the objective, stated (not derived term-by-term)

1. In high-dim space, convert distances to conditional probabilities that
   point $j$ would be picked as a neighbor of $i$, using a Gaussian
   centered at $i$: $p_{j|i}\propto \exp(-\|x_i-x_j\|^2/2\sigma_i^2)$.
   $\sigma_i$ is chosen per-point via a user-set **perplexity** parameter
   (roughly: "effective number of neighbors").
2. In the low-dim (2D/3D) embedding, define similar probabilities $q_{ij}$
   using a **Student-t distribution** (heavier tails — this is the "t" in
   t-SNE, and it's what prevents the "crowding problem" where moderate
   high-dim distances would otherwise all get crushed together in 2D).
3. Minimize the **KL divergence** $\sum_{i,j} p_{ij}\log\frac{p_{ij}}{q_{ij}}$
   between the two distributions via gradient descent on the 2D point
   positions.

Key intuitive facts worth stating explicitly (these are the common
misconceptions to warn students about):
- **Cluster sizes and inter-cluster distances in a t-SNE plot are not
  meaningful** — only local neighborhood relationships are preserved.
- Results depend heavily on **perplexity** (typically 5–50) — always show
  multiple perplexity values, never trust a single run.
- t-SNE has no `.transform()` for new data in the classical form (it's not
  a mapping, it's an optimization result for that specific dataset) —
  contrast explicitly with PCA, which does generalize to new points.

## 3. UMAP — the objective, stated

Built on ideas from topology (fuzzy simplicial sets) rather than
probability, but produces a conceptually similar output: a low-dimensional
layout optimizing a **cross-entropy** between high-dim and low-dim fuzzy
neighborhood graphs. Practical differences from t-SNE worth a comparison
table (not derivation):

| | t-SNE | UMAP |
|---|---|---|
| Preserves | local structure only | local + more global structure |
| Speed | slower, $O(n^2)$ naive (Barnes-Hut $O(n\log n)$) | faster, scales better |
| New data | no native transform | has `.transform()` for new points |
| Key parameter | perplexity | `n_neighbors`, `min_dist` |
| Theoretical basis | probability / KL divergence | algebraic topology |

## 4. Visualization (the heart of this notebook)

- Run PCA, t-SNE (perplexity ∈ {5, 30, 50}), and UMAP on the **same**
  dataset (`Iris.csv` or `sklearn.datasets.load_digits()` — digits is more
  visually convincing since classes are less separable in raw pixel space)
  side by side in one figure grid.
- Animate t-SNE's iterations converging (matplotlib
  `FuncAnimation` over stored intermediate embeddings —
  `sklearn.manifold.TSNE` doesn't expose these directly, so either use
  `openTSNE` which does, or manually loop a small number of gradient steps
  using the `n_iter` parameter across repeated calls to illustrate the
  *idea* of iterative refinement, clearly caveated as illustrative).
- A perplexity-sensitivity sweep: same data, 4 subplots at perplexity
  5/15/30/50, so students see how much the picture changes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap  # pip install umap-learn

digits = load_digits()
X, y = digits.data, digits.target

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, model) in zip(axes, [
        ("PCA", PCA(n_components=2)),
        ("t-SNE", TSNE(n_components=2, perplexity=30, random_state=0)),
        ("UMAP", umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0)),
    ]):
    proj = model.fit_transform(X)
    sc = ax.scatter(proj[:,0], proj[:,1], c=y, cmap="tab10", s=12)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("../_assets/pca_tsne_umap_comparison.png", dpi=130)

![pca_tsne_umap_comparison](../_assets/pca_tsne_umap_comparison.png)


## 5. When to use which

feeds `11_Algorithm_Selection_and_Comparison/05_compare_dimensionality_reduction.ipynb`

- Need an interpretable, generalizable, linear projection (e.g. as a
  preprocessing step before another model) → **PCA**.
- Need the best possible 2D visualization of cluster structure in a
  report/paper, and won't need to project new points later → **t-SNE**.
- Need a visualization that also scales to larger datasets and/or must
  transform new points later (e.g. in a production embedding pipeline) →
  **UMAP**.